In [20]:
import numpy as np
import pandas as pd
import polpo.preprocessing.dict as ppdict
import polpo.preprocessing.pd as ppd
from polpo.model_eval import (
    MeshEuclideanR2Score,
    MeshR2Score,
    MultiEvaluator,
    OlsPValues,
    PcaEvaluator,
    R2Score,
    ReconstructionError,
    ResultsExtender,
    VertexReconstructionError,
    collect_obj_regr_eval_results,
)
from polpo.models import ObjectRegressor, SupervisedEmbeddingRegressor
from polpo.preprocessing import PartiallyInitializedStep
from polpo.preprocessing.learning import DictsToXY
from polpo.preprocessing.load.pregnancy import (
    DenseMaternalCsvDataLoader,
    DenseMaternalMeshLoader,
)
from polpo.preprocessing.mesh.conversion import PvFromData
from polpo.preprocessing.mesh.io import FreeSurferReader
from polpo.preprocessing.mesh.registration import PvAlign
from polpo.sklearn.adapter import AdapterPipeline, EvaluatedModel
from polpo.sklearn.mesh import BiMeshesToVertices
from polpo.sklearn.np import BiFlattenButFirst
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import FunctionTransformer
from sklearn.metrics import r2_score
from scipy.stats import f

subject_id = "01"
pilot = subject_id == "01"

# Consistent structure names for display
structure_display_names = {
    "Hipp": "Hippocampus",
    "Amyg": "Amygdala",
    "Thal": "Thalamus-Proper",
    "Caud": "Caudate",
    "Puta": "Putamen",
    "Pall": "Pallidum",
    "Accu": "Accumbens",
}

# List of structures to loop through
structures = [
    "Thal",
    "Caud",
    "Puta",
    "Pall",
    "Hipp",
    "Amyg"
]

# Prepare an empty list to collect results
results_list = []
results_dict = {}

# Loop through each structure and hemisphere
for struct in structures:
    row_results = {}
    for left in [True, False]:
        try:
            # Load data
            csv_loader = DenseMaternalCsvDataLoader(pilot=pilot, subject_id=subject_id)
            df = csv_loader()

            # Preprocess predictor
            session_selector = ppd.DfIsInFilter("stage", ["post"], negate=True)
            predictor_selector = (
                session_selector + ppd.ColumnsSelector("gestWeek") + ppd.SeriesToDict()
            )
            x_dict = predictor_selector(df)
            # print(x_dict)
            train_dict = {k: v for k, v in x_dict.items() if 1 <= k <= 16}
            test_dict = {k: v for k, v in x_dict.items() if 17 <= k <= 19}

            # Load and preprocess meshes
            mesh_loader = DenseMaternalMeshLoader(
                subject_id=subject_id,
                as_dict=True,
                left=left,
                struct=struct,
                derivative="enigma",
            )
            mesh_reader = ppdict.DictMap(FreeSurferReader() + PvFromData())

            prep_pipe = PartiallyInitializedStep(
                Step=lambda **kwargs: ppdict.DictMap(PvAlign(**kwargs)),
                _target=lambda meshes: meshes[list(meshes.keys())[0]],
                max_iterations=500,
            )

            mesh_pipe = mesh_loader + mesh_reader + prep_pipe
            meshes = mesh_pipe()

            # Model and pipeline setup
            objs2y = AdapterPipeline(
                steps=[
                    BiMeshesToVertices(index=0),
                    FunctionTransformer(func=np.stack),
                    BiFlattenButFirst(),
                ]
            )

            model = SupervisedEmbeddingRegressor(
                EvaluatedModel(
                    PLSRegression(n_components=2),
                    MultiEvaluator(
                        [
                            ReconstructionError(),
                            VertexReconstructionError(prefix="vertex"),
                        ]
                    ),
                ),
                EvaluatedModel(
                    LinearRegression(),
                    MultiEvaluator([OlsPValues(), R2Score()]),
                ),
            )

            obj_model = EvaluatedModel(
                ObjectRegressor(model, objs2y),
                MultiEvaluator(
                    [MeshEuclideanR2Score(), MeshR2Score()],
                    extender=ResultsExtender(),
                ),
            )

            # Create dataset
            dataset_pipe = DictsToXY()
            X, meshes_ = dataset_pipe((x_dict, meshes))
            X_train, meshes_train = dataset_pipe((train_dict, meshes))
            X_test, meshes_test = dataset_pipe((test_dict, meshes))

            # Fit and evaluate
            # obj_model.fit(X, meshes_)
            obj_model.fit(X_train, meshes_train)
            predictions_test = obj_model.predict(X_test)

            # Evaluate on train and test sets
            eval_results = collect_obj_regr_eval_results(obj_model)

            # Flatten true and predicted meshes
            true_flat = np.stack([m.points.flatten() for m in meshes_test])
            pred_flat = np.stack([m.points.flatten() for m in predictions_test])

            # Compute global R² for test data
            r2_test = r2_score(true_flat.flatten(), pred_flat.flatten())

            # Extract p-values and R² from "regr-encoder"
            regr_encoder = eval_results["regr-encoder"]
            pvalues = eval_results["regr-regr"]["pvals"]
            r2_train = eval_results["obj_regr"]["featurewise_r2-mean"]
            r2_test = r2_test

            # Compute p-value for H0: R² = 0
            n = len(X_test)  # number of samples in test set
            k = 1  # single predictor (gestWeek)
            F_stat = (r2_test / k) / ((1 - r2_test) / (n - k - 1))
            p_value_r2 = 1 - f.cdf(F_stat, k, n - k - 1)

            # Format for display
            formatted_entry = f"p={pvalues[0].item():.3e}, R2(train)={r2_train:.4f}, R2(test)={r2_test:.4f}"
            row_results["left" if left else "right"] = formatted_entry

            # Store results
            results_list.append({
                "structure": struct,
                "left": left,
                "p-values": pvalues[0],
                "r2-train": r2_train,
                "r2-test": r2_test,
                "r2-test-p-value": p_value_r2
            })

        except Exception as e:
            print(f"Error processing {struct} {'left' if left else 'right'}: {e}")
            results_list.append({
                "structure": struct,
                "left": left,
                "p-values": np.nan,
                "r2-train": np.nan,
                "r2-test": np.nan,
                "r2-test-p-value": np.nan
            })

    # Store results with display name
    structure_name = structure_display_names.get(struct, struct)
    results_dict[structure_name] = row_results

# Convert to DataFrame for pretty display
final_df = pd.DataFrame.from_dict(results_dict, orient="index")
final_df.index.name = "Structure"
final_df.columns = ["Left", "Right"]

# Display final table
print(final_df)
final_df.to_csv("results_table_pls.csv")

# # Create a DataFrame for easier viewing
# results_df = pd.DataFrame(results_list)

INFO: Data has already been downloaded... using cached file ('/Users/sak/.herbrain/data/maternal/raw/28Baby_Hormones.csv').
INFO: Data has already been downloaded... using cached file ('/Users/sak/.herbrain/data/maternal/raw/28Baby_Hormones.csv').
INFO: Data has already been downloaded... using cached file ('/Users/sak/.herbrain/data/maternal/raw/28Baby_Hormones.csv').
INFO: Data has already been downloaded... using cached file ('/Users/sak/.herbrain/data/maternal/raw/28Baby_Hormones.csv').
INFO: Data has already been downloaded... using cached file ('/Users/sak/.herbrain/data/maternal/raw/28Baby_Hormones.csv').
INFO: Data has already been downloaded... using cached file ('/Users/sak/.herbrain/data/maternal/raw/28Baby_Hormones.csv').
INFO: Data has already been downloaded... using cached file ('/Users/sak/.herbrain/data/maternal/raw/28Baby_Hormones.csv').
INFO: Data has already been downloaded... using cached file ('/Users/sak/.herbrain/data/maternal/raw/28Baby_Hormones.csv').
INFO: Da

                                                           Left  \
Structure                                                         
Thalamus-Proper  p=4.769e-07, R2(train)=0.1129, R2(test)=0.9991   
Caudate          p=8.868e-05, R2(train)=0.0904, R2(test)=0.9998   
Putamen          p=6.517e-05, R2(train)=0.1603, R2(test)=0.9995   
Pallidum         p=8.393e-07, R2(train)=0.2157, R2(test)=0.9991   
Hippocampus      p=1.489e-06, R2(train)=0.1183, R2(test)=0.9994   
Amygdala         p=1.227e-03, R2(train)=0.1466, R2(test)=0.9909   

                                                          Right  
Structure                                                        
Thalamus-Proper  p=7.031e-07, R2(train)=0.1260, R2(test)=0.9995  
Caudate          p=3.950e-05, R2(train)=0.1455, R2(test)=0.9997  
Putamen          p=2.025e-06, R2(train)=0.1438, R2(test)=0.9998  
Pallidum         p=1.356e-06, R2(train)=0.2179, R2(test)=0.9998  
Hippocampus      p=1.948e-06, R2(train)=0.1078, R2(test)=0.9997  
A

In [3]:
# Display the table
import tabulate
print(tabulate.tabulate(results_df, headers='keys', tablefmt='psql'))

+----+-------------+--------+-------------+------------+-----------+-------------------+
|    | structure   | left   |    p-values |   r2-train |   r2-test |   r2-test-p-value |
|----+-------------+--------+-------------+------------+-----------+-------------------|
|  0 | Hipp        | True   | 2.30687e-05 |   0.220854 |  0.999447 |        0.014969   |
|  1 | Hipp        | False  | 2.30687e-05 |   0.220854 |  0.999668 |        0.0116013  |
|  2 | Amyg        | True   | 2.30687e-05 |   0.220854 |  0.990919 |        0.0607592  |
|  3 | Amyg        | False  | 2.30687e-05 |   0.220854 |  0.999886 |        0.00679144 |
|  4 | Thal        | True   | 2.30687e-05 |   0.220854 |  0.999111 |        0.0189805  |
|  5 | Thal        | False  | 2.30687e-05 |   0.220854 |  0.999451 |        0.014922   |
|  6 | Caud        | True   | 2.30687e-05 |   0.220854 |  0.999823 |        0.00847862 |
|  7 | Caud        | False  | 2.30687e-05 |   0.220854 |  0.999652 |        0.0118788  |
|  8 | Puta        | 

In [28]:
results_dict = {}

for struct in structures:
    row_results = {}
    for left in [True, False]:
        try:
            # Load data
            csv_loader = DenseMaternalCsvDataLoader(pilot=pilot, subject_id=subject_id)
            df = csv_loader()

            # Preprocess predictor
            session_selector = ppd.DfIsInFilter("stage", ["post"], negate=True)
            predictor_selector = (
                session_selector + ppd.ColumnsSelector("gestWeek") + ppd.SeriesToDict()
            )
            x_dict = predictor_selector(df)

            # Load and preprocess meshes
            mesh_loader = DenseMaternalMeshLoader(
                subject_id=subject_id,
                as_dict=True,
                left=left,
                struct=struct,
                derivative="enigma",
            )
            mesh_reader = ppdict.DictMap(FreeSurferReader() + PvFromData())

            prep_pipe = PartiallyInitializedStep(
                Step=lambda **kwargs: ppdict.DictMap(PvAlign(**kwargs)),
                _target=lambda meshes: meshes[list(meshes.keys())[0]],
                max_iterations=500,
            )

            mesh_pipe = mesh_loader + mesh_reader + prep_pipe
            meshes = mesh_pipe()

            # Create dataset: train/test split
            train_dict = {k: v for k, v in x_dict.items() if 1 <= k <= 16}
            test_dict = {k: v for k, v in x_dict.items() if 17 <= k <= 19}

            dataset_pipe = DictsToXY()
            X_all, meshes_all = dataset_pipe((x_dict, meshes))
            X_train, meshes_train = dataset_pipe((train_dict, meshes))
            X_test, meshes_test = dataset_pipe((test_dict, meshes))

            # Compute mesh volumes
            volumes_train = np.array([m.volume for m in meshes_train])
            volumes_test = np.array([m.volume for m in meshes_test])

            print(f"Structure: {struct}, left: {left}")
            print("Volumes train:", volumes_train)
            print("Volumes test:", volumes_test)

            # Fit simple linear regression: gestWeek -> volume
            regr = LinearRegression()
            regr.fit(X_train, volumes_train)

            # Predict on train and test
            pred_train = regr.predict(X_train)
            pred_test = regr.predict(X_test)

            # Compute R² for train and test
            r2_train = r2_score(volumes_train, pred_train)
            r2_test = r2_score(volumes_test, pred_test)

            # Compute p-value for H0: R² = 0 (test data)
            n = len(X_test)
            k = 1
            F_stat = (r2_test / k) / ((1 - r2_test) / (n - k - 1))
            p_value_r2 = 1 - f.cdf(F_stat, k, n - k - 1)

            # Format for display
            formatted_entry = f"p={p_value_r2:.3e}, R2(train)={r2_train:.4f}, R2(test)={r2_test:.4f}"
            row_results["left" if left else "right"] = formatted_entry

        except Exception as e:
            print(f"Error processing {struct} {'left' if left else 'right'}: {e}")
            row_results["left" if left else "right"] = "Error"

    # Store results with display name
    structure_name = structure_display_names.get(struct, struct)
    results_dict[structure_name] = row_results

# Convert to DataFrame for pretty display
final_df = pd.DataFrame.from_dict(results_dict, orient="index")
final_df.index.name = "Structure"
final_df.columns = ["Left", "Right"]

# Display final table
print("\nFinal volume regression results table:\n")
print(final_df)

# Save to CSV
final_df.to_csv("results_table_volume.csv")


INFO: Data has already been downloaded... using cached file ('/Users/sak/.herbrain/data/maternal/raw/28Baby_Hormones.csv').
INFO: Data has already been downloaded... using cached file ('/Users/sak/.herbrain/data/maternal/raw/28Baby_Hormones.csv').


Structure: Thal, left: True
Volumes train: [7605.37936252 7624.17314289 7587.78650967 7624.56721466 7627.03399119
 7575.33214542 7334.212782   7424.60608562 7399.12205749 7223.99552902
 7465.3720632  7276.2203299  7301.1121013  7314.20992266 7322.00791108
 7459.85450125]
Volumes test: [7335.55959303 7210.12787697 7202.69151856]


INFO: Data has already been downloaded... using cached file ('/Users/sak/.herbrain/data/maternal/raw/28Baby_Hormones.csv').


Structure: Thal, left: False
Volumes train: [7372.13559124 7451.10690396 7416.91719284 7414.26589267 7459.75378224
 7365.27610379 7061.29583821 7220.63350599 7219.8759827  6954.1917721
 7299.06477071 7061.35622582 7141.84763201 7089.47420828 7079.45027494
 7196.97679434]
Volumes test: [7199.38011501 6982.58603971 7043.51450765]


INFO: Data has already been downloaded... using cached file ('/Users/sak/.herbrain/data/maternal/raw/28Baby_Hormones.csv').


...

Structure: Pall, left: False
Volumes train: [1774.72796985 1797.05151212 1788.23909565 1810.40847248 1799.07796226
 1784.09029439 1703.85623232 1755.67169502 1761.92899143 1674.43082675
 1646.56246061 1676.50717068 1635.78503219 1662.18893527 1647.502737
 1603.01961475]
Volumes test: [1597.80580439 1594.1465411  1626.59298081]


INFO: Data has already been downloaded... using cached file ('/Users/sak/.herbrain/data/maternal/raw/28Baby_Hormones.csv').
INFO: Data has already been downloaded... using cached file ('/Users/sak/.herbrain/data/maternal/raw/28Baby_Hormones.csv').


Structure: Hipp, left: True
Volumes train: [3634.40074975 3719.00312939 3722.08652649 3669.57924397 3674.26687266
 3600.61761081 3428.87374706 3586.52243706 3577.82114748 3504.89174034
 3488.14452355 3453.90189894 3471.25522244 3461.40797685 3439.80363139
 3477.66323661]
Volumes test: [3449.31803415 3422.11571762 3455.30131711]
Structure: Hipp, left: False
Volumes train: [3830.06823945 3847.52583418 3886.153544   3825.24915947 3864.66265129
 3781.09240699 3564.55172784 3730.91022534 3767.32896049 3665.66586175
 3628.0828766  3642.12495387 3648.02100504 3632.95563456 3556.31217045
 3624.36617354]
Volumes test: [3603.47214655 3553.79113292 3569.92226901]


INFO: Data has already been downloaded... using cached file ('/Users/sak/.herbrain/data/maternal/raw/28Baby_Hormones.csv').


Structure: Amyg, left: True
Volumes train: [1488.2393153  1516.16533174 1548.00209323 1538.30477528 1539.60012396
 1544.30699934 1421.00589293 1498.59015942 1483.13298933 1410.87202976
 1416.89393241 1452.92410669 1443.21570748 1420.39696021 1402.18786436
 1420.25296446]
Volumes test: [1382.48676758 1363.5288389  1365.31025482]
Structure: Amyg, left: False
Volumes train: [1488.82095824 1512.45100229 1530.29424283 1531.62091058 1523.36096662
 1505.11075284 1406.98249448 1495.77358297 1475.3692399  1403.78401366
 1439.37679781 1441.5641941  1398.52821569 1422.11150826 1421.75906116
 1431.30225417]
Volumes test: [1408.04380094 1379.97751613 1421.23102631]

Final volume regression results table:

                                                             Left  \
Structure                                                           
Thalamus-Proper    p=6.296e-01, R2(train)=0.5864, R2(test)=0.3021   
Caudate           p=1.000e+00, R2(train)=0.7151, R2(test)=-0.2229   
Putamen           p=1.

In [29]:
for struct in structures:
    row_results = {}
    for left in [True, False]:
        try:
            # Load data
            csv_loader = DenseMaternalCsvDataLoader(pilot=pilot, subject_id=subject_id)
            df = csv_loader()

            # Preprocess predictor
            session_selector = ppd.DfIsInFilter("stage", ["post"], negate=True)
            predictor_selector = (
                session_selector + ppd.ColumnsSelector("gestWeek") + ppd.SeriesToDict()
            )
            x_dict = predictor_selector(df)

            # Load and preprocess meshes
            mesh_loader = DenseMaternalMeshLoader(
                subject_id=subject_id,
                as_dict=True,
                left=left,
                struct=struct,
                derivative="enigma",
            )
            mesh_reader = ppdict.DictMap(FreeSurferReader() + PvFromData())

            prep_pipe = PartiallyInitializedStep(
                Step=lambda **kwargs: ppdict.DictMap(PvAlign(**kwargs)),
                _target=lambda meshes: meshes[list(meshes.keys())[0]],
                max_iterations=500,
            )

            mesh_pipe = mesh_loader + mesh_reader + prep_pipe
            meshes = mesh_pipe()

            # Create dataset: train/test split
            train_dict = {k: v for k, v in x_dict.items() if 1 <= k <= 16}
            test_dict = {k: v for k, v in x_dict.items() if 17 <= k <= 19}

            dataset_pipe = DictsToXY()
            X_all, meshes_all = dataset_pipe((x_dict, meshes))
            X_train, meshes_train = dataset_pipe((train_dict, meshes))
            X_test, meshes_test = dataset_pipe((test_dict, meshes))

            # Compute mesh volumes
            volumes_train = np.array([m.volume for m in meshes_train])
            volumes_test = np.array([m.volume for m in meshes_test])

            print(f"Structure: {struct}, left: {left}")
            print("Volumes train:", volumes_train)
            print("Volumes test:", volumes_test)

            # Fit simple linear regression: gestWeek -> volume
            regr = LinearRegression()
            regr.fit(X_train, volumes_train)

            # Predict on train and test
            pred_train = regr.predict(X_train)
            pred_test = regr.predict(X_test)

            # Compute R² for train and test
            r2_train = r2_score(volumes_train, pred_train)
            r2_test = r2_score(volumes_test, pred_test)

            # Compute p-value for H0: R² = 0 (test data)
            n = len(X_test)
            k = 1
            F_stat = (r2_test / k) / ((1 - r2_test) / (n - k - 1))
            p_value_r2 = 1 - f.cdf(F_stat, k, n - k - 1)

            # Apply Bonferroni correction for multiple comparisons
            corrected_p_value = min(p_value_r2 * 28, 1.0)

            # Format for display
            formatted_entry = f"p={corrected_p_value:.3e} (corrected), R2(train)={r2_train:.4f}, R2(test)={r2_test:.4f}"
            row_results["left" if left else "right"] = formatted_entry

        except Exception as e:
            print(f"Error processing {struct} {'left' if left else 'right'}: {e}")
            row_results["left" if left else "right"] = "Error"

    # Store results with display name
    structure_name = structure_display_names.get(struct, struct)
    results_dict[structure_name] = row_results

# Convert to DataFrame for pretty display
final_df = pd.DataFrame.from_dict(results_dict, orient="index")
final_df.index.name = "Structure"
final_df.columns = ["Left", "Right"]

# Display final table
print("\nFinal volume regression results table:\n")
print(final_df)

# Save to CSV
final_df.to_csv("results_table_volume_corrected.csv")


INFO: Data has already been downloaded... using cached file ('/Users/sak/.herbrain/data/maternal/raw/28Baby_Hormones.csv').
INFO: Data has already been downloaded... using cached file ('/Users/sak/.herbrain/data/maternal/raw/28Baby_Hormones.csv').


Structure: Thal, left: True
Volumes train: [7605.37936252 7624.17314289 7587.78650967 7624.56721466 7627.03399119
 7575.33214542 7334.212782   7424.60608562 7399.12205749 7223.99552902
 7465.3720632  7276.2203299  7301.1121013  7314.20992266 7322.00791108
 7459.85450125]
Volumes test: [7335.55959303 7210.12787697 7202.69151856]


INFO: Data has already been downloaded... using cached file ('/Users/sak/.herbrain/data/maternal/raw/28Baby_Hormones.csv').


Structure: Thal, left: False
Volumes train: [7372.13559124 7451.10690396 7416.91719284 7414.26589267 7459.75378224
 7365.27610379 7061.29583821 7220.63350599 7219.8759827  6954.1917721
 7299.06477071 7061.35622582 7141.84763201 7089.47420828 7079.45027494
 7196.97679434]
Volumes test: [7199.38011501 6982.58603971 7043.51450765]


INFO: Data has already been downloaded... using cached file ('/Users/sak/.herbrain/data/maternal/raw/28Baby_Hormones.csv').


...

Structure: Pall, left: False
Volumes train: [1774.72796985 1797.05151212 1788.23909565 1810.40847248 1799.07796226
 1784.09029439 1703.85623232 1755.67169502 1761.92899143 1674.43082675
 1646.56246061 1676.50717068 1635.78503219 1662.18893527 1647.502737
 1603.01961475]
Volumes test: [1597.80580439 1594.1465411  1626.59298081]


INFO: Data has already been downloaded... using cached file ('/Users/sak/.herbrain/data/maternal/raw/28Baby_Hormones.csv').
INFO: Data has already been downloaded... using cached file ('/Users/sak/.herbrain/data/maternal/raw/28Baby_Hormones.csv').


Structure: Hipp, left: True
Volumes train: [3634.40074975 3719.00312939 3722.08652649 3669.57924397 3674.26687266
 3600.61761081 3428.87374706 3586.52243706 3577.82114748 3504.89174034
 3488.14452355 3453.90189894 3471.25522244 3461.40797685 3439.80363139
 3477.66323661]
Volumes test: [3449.31803415 3422.11571762 3455.30131711]
Structure: Hipp, left: False
Volumes train: [3830.06823945 3847.52583418 3886.153544   3825.24915947 3864.66265129
 3781.09240699 3564.55172784 3730.91022534 3767.32896049 3665.66586175
 3628.0828766  3642.12495387 3648.02100504 3632.95563456 3556.31217045
 3624.36617354]
Volumes test: [3603.47214655 3553.79113292 3569.92226901]


INFO: Data has already been downloaded... using cached file ('/Users/sak/.herbrain/data/maternal/raw/28Baby_Hormones.csv').


Structure: Amyg, left: True
Volumes train: [1488.2393153  1516.16533174 1548.00209323 1538.30477528 1539.60012396
 1544.30699934 1421.00589293 1498.59015942 1483.13298933 1410.87202976
 1416.89393241 1452.92410669 1443.21570748 1420.39696021 1402.18786436
 1420.25296446]
Volumes test: [1382.48676758 1363.5288389  1365.31025482]
Structure: Amyg, left: False
Volumes train: [1488.82095824 1512.45100229 1530.29424283 1531.62091058 1523.36096662
 1505.11075284 1406.98249448 1495.77358297 1475.3692399  1403.78401366
 1439.37679781 1441.5641941  1398.52821569 1422.11150826 1421.75906116
 1431.30225417]
Volumes test: [1408.04380094 1379.97751613 1421.23102631]

Final volume regression results table:

                                                              Left  \
Structure                                                            
Thalamus-Proper  p=1.000e+00 (corrected), R2(train)=0.5864, R2(...   
Caudate          p=1.000e+00 (corrected), R2(train)=0.7151, R2(...   
Putamen          p